# Master file

This file contains all 2d resnet plots

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from matplotlib.colors import to_rgb
from matplotlib.colors import LinearSegmentedColormap


# Model and Training Params

In [ ]:
# Basic Model Params relevant for training

input_dim = 2
hidden_dim = 2
output_dim = 1
activation = 'tanh' #'relu' and 'tanh' are supported

# Training Params
load_file = None
cross_entropy = True #True supported with binary classification only, False needs accuracy fix

subfolder = 'master_file'
import os
if not os.path.exists(subfolder):
    os.makedirs(subfolder)

In [ ]:
import models.training
from models.training import make_circles_uniform

# Generate training data

# Set random seed for reproducibility
seed = np.random.randint(1000)
# seed = 163
np.random.seed(seed)
torch.manual_seed(seed)

#footnnote to display on plots to make sure that plots and model/trainign params do not get confused

n_points = 2000 #number of points in the dataset

inner_radius = 0.5
outer_radius = 1
buffer = 0.2

import importlib
importlib.reload(models.training) # Reload the module

train_loader, test_loader = make_circles_uniform(output_dim = 1, n_samples = n_points, inner_radius = 0.5, outer_radius = 1.0, buffer = 0.1, cross_entropy=cross_entropy, seed = seed, filename = subfolder + '/circles_dataset')

In [ ]:
for input, label in train_loader:
    print(input[:5], label[:5])
    break

# Define model and training


In [ ]:
# to reload models.resnet module after changes without restarting the kernel
import importlib
import models.resnets
import models.training
importlib.reload(models.resnets) # Reload the module
importlib.reload(models.training) # Reload the module
from models.resnets import ResNet
from models.training import train_until_threshold, plot_loss_curve


# Neural ODE Regime

In [ ]:
num_hidden = 20 # number of hidden layers. The total network has additionl 2 layers: input to hidden and hidden to output
residual_param = 2/num_hidden
skip_param = 1
batch_norm = True

num_epochs = 100

footnote = f'num_hidden={num_hidden}, hidden_dim={hidden_dim}, output_dim={output_dim}, act={activation}, seed={seed}, ce={cross_entropy}'




filename = subfolder + '/2d_node_regime'

# Train models
model_base, acc_base, losses_base = train_until_threshold(ResNet,
    train_loader, test_loader,
    load_file = filename, epochs = num_epochs, max_retries=10, threshold=0.9,
    input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_hidden=num_hidden, skip_param=skip_param, residual_param = residual_param, activation=activation, batchnorm = batch_norm, save_path = filename
)

# train_model(model_base, train_loader, test_loader, epochs=num_epochs, save_path = filename)

plot_loss_curve(losses_base, title=f"Base Model Loss Curve", filename = filename + '_loss')

In [ ]:
model_base.parameters

In [ ]:
import importlib
import plots.plots 
from plots.plots import plot_decision_boundary, plot_level_sets
importlib.reload(plots.plots) # Reload the module


X_test, y_test = next(iter(test_loader))
plot_decision_boundary(model_base, X_test, y_test, show=True, file_name = filename + '_pred', footnote = footnote, amount_levels= 100, title = f'$\epsilon$ = {skip_param}, $\delta$ = {residual_param}')
plot_level_sets(model_base, show=True, file_name= filename + '_ls', footnote = footnote, amount_levels= 20, title = f'$\epsilon$ = {skip_param}, $\delta$ = {residual_param}')



# MLP Constant width 2 

In [ ]:
# Model Params 
num_hidden = 6 # number of hidden layers. The total network has additionl 2 layers: input to hidden and hidden to output
input_dim = 2
hidden_dim = 2
output_dim = 1
activation = 'tanh' #'relu' and 'tanh' are supported

# Training Params
cross_entropy = True #True supported with binary classification only, False needs accuracy fix
num_epochs = 300





In [ ]:
skip_param = 0.1
residual_param = 1

footnote = f'num_hidden={num_hidden}, hidden_dim={hidden_dim}, output_dim={output_dim}, act={activation}, seed={seed}, ce={cross_entropy}'

filename = subfolder + '/2d_mlp_regime'


# Train models
model_base, acc_base, losses_base = train_until_threshold(ResNet,
    train_loader, test_loader,
    load_file = filename, max_retries=10, threshold=0.90,
    input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_hidden=num_hidden, skip_param=skip_param, residual_param = residual_param, activation=activation
)

plot_loss_curve(losses_base, title=f"Base Model Loss Curve", filename = filename + '_loss')

In [ ]:
model_base.parameters

In [ ]:
import importlib
import plots.plots 
from plots.plots import plot_decision_boundary, plot_level_sets
importlib.reload(plots.plots) # Reload the module

X_test, y_test = next(iter(test_loader))
plot_decision_boundary(model_base, X_test, y_test, show=True, file_name= filename + '_pred', footnote = footnote, amount_levels= 100, title = f'$\epsilon$ = {skip_param}, $\delta$ = {residual_param}')
plot_level_sets(model_base, show=True, file_name= filename + str(num_hidden) + '_ls', footnote = footnote, amount_levels= 20, title = f'$\epsilon$ = {skip_param}, $\delta$ = {residual_param}')



# Augmented model: width 3

In [ ]:
hidden_dim = 3
num_hidden = 1

skip_param = 0

footnote = f'num_hidden={num_hidden}, hidden_dim={hidden_dim}, output_dim={output_dim}, act={activation}, seed={seed}, ce={cross_entropy}'

seed = 288

filename = subfolder + '/2d_augmented'

model_aug, acc_aug, losses_aug = train_until_threshold(ResNet,
    train_loader, test_loader,
    load_file = filename, save_path = filename, max_retries=5, threshold=0.95, seed=seed,
    input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_hidden=num_hidden, skip_param=skip_param, activation=activation
)

plot_loss_curve(losses_aug, title=f"Augmented Model Loss Curve", filename = subfolder + '/ff6hidden_aug')

In [ ]:
X_test, y_test = next(iter(test_loader))
footnote = f'num_hidden={num_hidden}, hidden_dim={hidden_dim}, output_dim={output_dim}, act={activation}, seed={seed}, ce={cross_entropy}'
plot_decision_boundary(model_aug, X_test, y_test, show=True, file_name= subfolder + '/ffaugcirc' + str(num_hidden), footnote = footnote, amount_levels= 100, title = f'$\epsilon$ = {skip_param}, $\delta$ = {residual_param}')
plot_level_sets(model_aug, show=True, file_name= subfolder + '/ffaugcirc_contour' + str(num_hidden), footnote = footnote, amount_levels= 20, title = f'Augmented MLP')



# ResNet model

In [ ]:
skip_param = 1 # this sets model from feed forward to residual network
residual_param = 1

num_hidden = 10
hidden_dim = 2

footnote = f'num_hidden={num_hidden}, hidden_dim={hidden_dim}, output_dim={output_dim}, act={activation}, seed={seed}, ce={cross_entropy}'

filename = subfolder + '/ff6_res'

model_res, acc_res, losses_res = train_until_threshold(ResNet,
    train_loader, test_loader,
    load_file = filename, save_path = filename, max_retries=5, threshold=0.95,
    input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_hidden=num_hidden, skip_param=skip_param, residual_param = residual_param, activation=activation,
)

plot_loss_curve(losses_res, title=f"ResNet Model Loss Curve", filename = subfolder + '/ff6_res')


In [ ]:
X_test, y_test = next(iter(test_loader))
plot_decision_boundary(model_res, X_test, y_test, show=True, file_name= filename + '_pred', footnote = footnote, amount_levels= 100, title = f'$\epsilon$ = {skip_param}, $\delta$ = {residual_param}')
plot_level_sets(model_res, show=True, file_name= filename + '_ls', footnote = footnote, amount_levels= 10, plotrange=[-2.5,2.5], title = f'$\epsilon$ = {skip_param}, $\delta$ = {residual_param}')

# XOR data set

In [ ]:
importlib.reload(models.training)
from models.training import make_xor_new

train_loader, test_loader = make_xor_new(1, noise = 0., filename = subfolder + '/xor_dataset_new')

In [ ]:
# Model Params
num_hidden = 6 # number of hidden layers. The total network has additionl 2 layers: input to hidden and hidden to output
input_dim = 2
hidden_dim = 2
output_dim = 1
activation = 'tanh' #'relu' and 'tanh' are supported


skip_param = 0.1
residual_param = 1

# Training Params
cross_entropy = True #True supported with binary classification only
num_epochs = 300



In [ ]:
footnote = f'num_hidden={num_hidden}, hidden_dim={hidden_dim}, output_dim={output_dim}, act={activation}, seed={seed}, ce={cross_entropy}'

filename = subfolder + '/new_xor_mlp_regime'

model_base, acc_base, losses_base = train_until_threshold(ResNet,
    train_loader, test_loader,
    load_file = filename, max_retries=10, threshold=0.95,
    input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_hidden=num_hidden, skip_param=0, activation=activation, save_path=filename
)

plot_loss_curve(losses_base, title=f"Base Model Loss Curve", filename = filename)

In [ ]:
X_test, y_test = next(iter(test_loader))
plot_decision_boundary(model_base, X_test, y_test, show=True, file_name= filename + '_pred', footnote = footnote, amount_levels= 100, title = f'$\epsilon$ = {skip_param}, $\delta$ = {residual_param}', plotrange=[-1.2,1.2])
plot_level_sets(model_base, show=True, file_name= filename + '_ls', footnote = footnote, amount_levels= 20, title = f'$\epsilon$ = {skip_param}, $\delta$ = {residual_param}', plotrange=[-1.2,1.2])

In [ ]:
hidden_dim = 3
num_hidden = 2

skip_param = 0
residual_param = 1

filename = subfolder + '/new_xor_aug_mlp'


footnote = f'num_hidden={num_hidden}, hidden_dim={hidden_dim}, output_dim={output_dim}, act={activation}, seed={seed}, ce={cross_entropy}'

model_wide, acc_wide, losses_wide = train_until_threshold(ResNet,
    train_loader, test_loader,
    load_file = filename, max_retries=5, threshold=0.95, epochs = 100,
    input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_hidden=num_hidden, skip_param=skip_param, residual_param = residual_param, activation=activation, save_path=filename
)

plot_loss_curve(losses_wide, title=f"Wide Model Loss Curve", filename = filename)

In [ ]:
X_test, y_test = next(iter(test_loader))
plot_decision_boundary(model_wide, X_test, y_test, show=True, file_name= filename + '_pred', footnote = footnote, amount_levels= 100, title = f'Augmented MLP', plotrange=[-1.2,1.2])
plot_level_sets(model_wide, show=True, file_name= filename + '_ls', footnote = footnote, amount_levels= 20, title = f'Augmented MLP', plotrange=[-1.2,1.2])

In [ ]:
skip_param = 1 # this sets model from feed forward to residual network
residual_param = 1

num_hidden = 10
hidden_dim = 2

num_epochs = 100

footnote = f'num_hidden={num_hidden}, hidden_dim={hidden_dim}, output_dim={output_dim}, act={activation}, seed={seed}, ce={cross_entropy}'

filename = subfolder + '/xor_res'

model_res, acc_res, losses_res = train_until_threshold(ResNet,
    train_loader, test_loader,
    load_file = None, save_path = filename, max_retries=5, threshold=0.95, epochs = num_epochs,
    input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_hidden=num_hidden, skip_param=skip_param, residual_param = residual_param, activation=activation,
)

plot_loss_curve(losses_res, title=f"ResNet Model Loss Curve", filename = filename + '_loss')


In [ ]:
X_test, y_test = next(iter(test_loader))
plot_decision_boundary(model_res, X_test, y_test, show=True, file_name= filename + '_pred', footnote = footnote, amount_levels= 100, title = f'$\epsilon$ = {skip_param}, $\delta$ = {residual_param}', plotrange=[-1.2,1.2])
plot_level_sets(model_res, show=True, file_name= filename + '_ls', footnote = footnote, amount_levels= 20, title = f'$\epsilon$ = {skip_param}, $\delta$ = {residual_param}', plotrange=[-1.2,1.2])